# LoRACM · Parametric imaging

Display posterior summaries from Scenario 1 on a verified coronal-slice geometry. Run inference first. The notebook preserves the voxel order and applies no smoothing or clipping to the estimates.

The supplied geometry is a **candidate derived from the original notebook**. Verify its spatial correspondence, orientation and spacing before setting the two verification flags in its JSON sidecar. This notebook will stop until then.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'common.py').exists():
    raise RuntimeError('Launch this notebook from LoRACM or LoRACM/notebooks')
sys.path.insert(0, str(ROOT))
from common import reconstruct_map

PREDICTIONS = ROOT / 'runs/scenario1/predictions.npz'
GEOMETRY = ROOT / 'data/real/Sub001_slice75_geometry.npz'
METADATA = ROOT / 'data/real/Sub001_slice75_geometry.json'
OUTPUT_DIR = PREDICTIONS.parent / 'figures'
STATISTIC = 'median'  # mean, median, or std
PARAMETER = 'Ki'      # Ki, K1, k2, k3, k4, Vb
VMAX = 0.06          # display limit only; use None for the finite-data maximum


## Validate the spatial mapping

The HDF5 column identifiers must match the released geometry identifiers in the same order. A prefix is permitted for a debugging run using `--max-voxels`; missing voxels remain transparent. Neither identifiers nor equal voxel counts independently prove anatomical correctness.

In [ ]:
metadata = json.loads(METADATA.read_text())
if not metadata.get('mapping_verified') or not metadata.get('orientation_and_spacing_verified'):
    raise ValueError('Author verification of mapping, orientation and spacing is required. See docs/RELEASE_REVIEW.md.')
with np.load(PREDICTIONS, allow_pickle=False) as f:
    predictions = {k: f[k] for k in f.files}
with np.load(GEOMETRY, allow_pickle=False) as f:
    geometry = {k: f[k] for k in f.files}
count = len(predictions['voxel_columns'])
if not np.array_equal(predictions['voxel_columns'], geometry['voxel_columns'][:count]):
    raise ValueError('Prediction order does not match the geometry column mapping')
run_metadata = json.loads(str(predictions['metadata_json']))
if run_metadata.get('time_mismatch_allowed'):
    print('EXPLORATORY RUN: training and observed time grids differ.')
if count != len(geometry['voxel_columns']):
    print(f'DEBUGGING SUBSET: showing {count} of {len(geometry["voxel_columns"])} voxels.')
print('Minimum Ki valid sample fraction:', predictions['Ki_valid_fraction'].min())
print('Average negative sample fractions [K1, k2, k3, k4, Vb]:', predictions['params_negative_fraction'].mean(axis=0))


## Display and export

Kᵢ is calculated samplewise as K₁k₃/(k₂+k₃). Means, medians and posterior standard deviations are distinct summaries. Near-zero denominator exclusions are recorded by inference; inspect their fractions and negative sample fractions above. Colour limits affect the display only.

In [ ]:
if STATISTIC not in ('mean', 'median', 'std'):
    raise ValueError('Choose mean, median or std')
if PARAMETER == 'Ki':
    values = predictions[f'Ki_{STATISTIC}']
else:
    names = predictions['parameter_names'].tolist()
    values = predictions[f'params_{STATISTIC}'][:, names.index(PARAMETER)]
image = reconstruct_map(values, geometry['mask'], geometry['voxel_indices'][:count])
units = {'K1': 'mL min⁻¹ cm⁻³', 'Ki': 'mL min⁻¹ cm⁻³',
         'k2': 'min⁻¹', 'k3': 'min⁻¹', 'k4': 'min⁻¹', 'Vb': 'dimensionless'}
fig, ax = plt.subplots(figsize=(5, 10), layout='constrained')
cmap = plt.colormaps['inferno'].copy()
cmap.set_bad('#eeeeee')
artist = ax.imshow(image, cmap=cmap, origin=metadata['origin'],
                   aspect=metadata['pixel_aspect_ratio'], interpolation='nearest', vmin=0, vmax=VMAX)
ax.set_title(f'{PARAMETER} · posterior {STATISTIC}', pad=12)
ax.set_axis_off()
fig.colorbar(artist, ax=ax, shrink=0.65, label=units[PARAMETER])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output = OUTPUT_DIR / f'{PARAMETER}_{STATISTIC}.png'
fig.savefig(output, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved {output}')
